In [1]:
import xarray as xr
import numpy as np
import dask
from dask.distributed import Client
from dask_jobqueue import PBSCluster

# PBS

In [9]:
cluster = PBSCluster(
    cores=1,
    memory='32GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='1:00:00'
)
cluster.scale(jobs=10)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.174:41465,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# data import

In [2]:
path='/glade/work/acruz/Caribbean_Heat_data/ERA5/'

In [3]:
hi_ds = xr.open_zarr(path+'hourly_HI')
hi_ds

<xarray.Dataset> Size: 19GB
Dimensions:    (time: 755304, latitude: 61, longitude: 101)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 488B 10.0 10.25 10.5 10.75 ... 24.5 24.75 25.0
  * longitude  (longitude) float64 808B -85.0 -84.75 -84.5 ... -60.25 -60.0
Data variables:
    HI_hourly  (time, latitude, longitude) float32 19GB dask.array<chunksize=(22888, 61, 101), meta=np.ndarray>

In [4]:
rh_ds = xr.open_dataset(path+'RH_during_peak_HI.nc')
rh_ds

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


<xarray.Dataset> Size: 1GB
Dimensions:           (time: 31471, latitude: 82, longitude: 121)
Coordinates:
  * time              (time) datetime64[ns] 252kB 1940-01-01 ... 2026-02-28
  * latitude          (latitude) float64 656B 7.75 8.0 8.25 ... 27.5 27.75 28.0
  * longitude         (longitude) float64 968B -89.0 -88.75 ... -59.25 -59.0
Data variables:
    RH_during_HIdmax  (time, latitude, longitude) float32 1GB ...

In [5]:
t_ds = xr.open_dataset(path+'T_during_peak_HI.nc')
t_ds

<xarray.Dataset> Size: 1GB
Dimensions:          (time: 31471, latitude: 82, longitude: 121)
Coordinates:
  * time             (time) datetime64[ns] 252kB 1940-01-01 ... 2026-02-28
  * latitude         (latitude) float64 656B 7.75 8.0 8.25 ... 27.5 27.75 28.0
  * longitude        (longitude) float64 968B -89.0 -88.75 ... -59.25 -59.0
Data variables:
    T_during_HIdmax  (time, latitude, longitude) float32 1GB ...

# monthly clim

In [6]:
rh_clim = rh_ds.groupby('time.month').mean()
rh_clim

<xarray.Dataset> Size: 478kB
Dimensions:           (month: 12, latitude: 82, longitude: 121)
Coordinates:
  * month             (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude          (latitude) float64 656B 7.75 8.0 8.25 ... 27.5 27.75 28.0
  * longitude         (longitude) float64 968B -89.0 -88.75 ... -59.25 -59.0
Data variables:
    RH_during_HIdmax  (month, latitude, longitude) float32 476kB 95.0 ... 92.09

In [8]:
t_clim = t_ds.groupby('time.month').mean()
t_clim

<xarray.Dataset> Size: 478kB
Dimensions:          (month: 12, latitude: 82, longitude: 121)
Coordinates:
  * month            (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude         (latitude) float64 656B 7.75 8.0 8.25 ... 27.5 27.75 28.0
  * longitude        (longitude) float64 968B -89.0 -88.75 ... -59.25 -59.0
Data variables:
    T_during_HIdmax  (month, latitude, longitude) float32 476kB 299.3 ... 295.9

## export

In [10]:
t_clim.to_netcdf(path+'T_clim_during_HIdmax.nc')

In [11]:
rh_clim.to_netcdf(path+'RH_clim_during_HIdmax.nc')

# quit

In [12]:
client.shutdown()